# Validation: Synthetic data with known ground truth

Generates a realistic multi-channel recording with:
- **White Gaussian noise** as the background
- **Planted oscillatory bursts** (Gaussian-windowed sinusoids) at known times,
  channels, frequencies, and durations

Runs the full pipeline and measures detection accuracy against the
ground-truth labels.

**Note on expected results:** The `EventDetector` uses an exponential moving
average (EMA, α=0.01) to track the running mean and variance of the
amplitude envelope. This converges slowly — after 50 chunks the initial
values still account for ~60% of the estimate — which leads to an
underestimated baseline and inflated false-positive rate on stationary
noise. Recall is typically good (70–90%) while precision is lower. This
is a known limitation of the current EMA approach; future work may
replace it with a batch baseline or faster-converging estimator.

In [ ]:
import numpy as np
import tempfile, os
from dataclasses import dataclass

# ============================================================
# 1. Synthetic data generation
# ============================================================

SAMPLE_RATE = 30000.0
N_CHANNELS = 4
DURATION = 30.0  # seconds
RNG_SEED = 42


@dataclass
class PlantedEvent:
    """Ground-truth label for a planted oscillatory burst."""
    channel: int
    onset: float      # seconds
    duration: float   # seconds
    frequency: float  # Hz
    snr: float        # amplitude relative to noise std


def inject_burst(signal, event, sample_rate):
    """Add a Gaussian-windowed sinusoidal burst to the signal."""
    start = int(event.onset * sample_rate)
    stop = int((event.onset + event.duration) * sample_rate)
    n = stop - start
    t = np.arange(n) / sample_rate
    # Gaussian window — 3-sigma fits within the burst
    sigma = event.duration / 6.0
    t_centre = event.duration / 2.0
    window = np.exp(-0.5 * ((t - t_centre) / sigma) ** 2)
    burst = event.snr * window * np.sin(2 * np.pi * event.frequency * t)
    signal[event.channel, start:stop] += burst


def generate_synthetic_recording(rng):
    """Build a full synthetic recording with planted events.

    Events are spaced >= 2.5s apart and start after 5s to clear the
    detector's warmup window (15 chunks x 0.2s = 3s).
    """
    n_samples = int(DURATION * SAMPLE_RATE)
    signal = rng.standard_normal((N_CHANNELS, n_samples))

    ground_truth = []
    for ch in range(N_CHANNELS):
        n_events = rng.integers(3, 6)
        onsets = np.sort(rng.uniform(5.0, DURATION - 2.0, size=n_events))
        # Enforce minimum 2.5s spacing
        filtered = [onsets[0]]
        for t in onsets[1:]:
            if t - filtered[-1] >= 2.5:
                filtered.append(t)

        for onset in filtered:
            ev = PlantedEvent(
                channel=ch,
                onset=round(onset, 4),
                duration=round(rng.uniform(0.05, 0.12), 4),
                frequency=round(rng.uniform(110, 180), 1),
                snr=round(rng.uniform(12.0, 25.0), 1),
            )
            inject_burst(signal, ev, SAMPLE_RATE)
            ground_truth.append(ev)

    return signal, ground_truth


rng = np.random.default_rng(RNG_SEED)
signal, ground_truth = generate_synthetic_recording(rng)

print(f"Generated {DURATION:.0f}s recording: {N_CHANNELS} channels @ {SAMPLE_RATE:.0f} Hz")
print(f"Planted {len(ground_truth)} events across {N_CHANNELS} channels")
print()
for ev in ground_truth:
    print(f"  ch={ev.channel} t={ev.onset:6.3f}s  dur={ev.duration*1000:5.1f}ms  "
          f"freq={ev.frequency:5.1f}Hz  snr={ev.snr:5.1f}")

In [ ]:
# ============================================================
# 2. Run the pipeline
# ============================================================

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import WaveletConvolution, EventDetector, PowerEstimator

path = os.path.join(tempfile.mkdtemp(), "synthetic.npz")
np.savez(path, continuous=signal, sample_rate=SAMPLE_RATE)

events = Pipeline(
    source=FileSource(path),
    modules=[
        WaveletConvolution(freq_min=10, freq_max=250, n_freqs=30),
        PowerEstimator(),
        EventDetector(
            event_type=EventType.RIPPLE,
            freq_range=(80, 250),
            threshold_std=4.0,
            min_duration=0.025,
            cooldown=0.15,
            warmup_chunks=15,  # 15 x 0.2s = 3s baseline
        ),
    ],
    config=PipelineConfig(
        sample_rate=SAMPLE_RATE,
        n_channels=N_CHANNELS,
        chunk_duration=0.2,
        buffer_duration=10.0,
    ),
).run_offline()

print(f"Detected {len(events)} events")
for e in events[:10]:
    print(f"  {e.event_type.name} ch={e.channel_id} "
          f"t={e.timestamp:.3f}s dur={e.duration:.3f}s "
          f"peak={e.metadata.get('peak_amplitude', 0):.2f}")
if len(events) > 10:
    print(f"  ... and {len(events) - 10} more")

In [ ]:
# ============================================================
# 3. Match detected events to ground truth
# ============================================================

TIME_TOLERANCE = 0.15  # max onset offset in seconds


def match_events(detected, ground_truth, tolerance):
    """Greedy nearest-neighbour matching by channel and time.

    Each ground-truth event is matched to the closest unmatched
    detected event on the same channel within the tolerance window.

    Returns: (matched_pairs, false_positives, false_negatives)
    where matched_pairs = [(gt_event, det_event, time_error), ...]
    """
    det_by_ch = {}
    for i, e in enumerate(detected):
        det_by_ch.setdefault(e.channel_id, []).append((i, e.timestamp))

    matched = []
    used_det = set()
    unmatched_gt = []

    for gt in ground_truth:
        candidates = det_by_ch.get(gt.channel, [])
        best_idx, best_err = None, tolerance + 1
        for det_idx, det_t in candidates:
            if det_idx in used_det:
                continue
            err = abs(det_t - gt.onset)
            if err < best_err:
                best_err = err
                best_idx = det_idx

        if best_idx is not None and best_err <= tolerance:
            matched.append((gt, detected[best_idx], best_err))
            used_det.add(best_idx)
        else:
            unmatched_gt.append(gt)

    false_positives = [e for i, e in enumerate(detected) if i not in used_det]
    return matched, false_positives, unmatched_gt


matched, fp, fn = match_events(events, ground_truth, TIME_TOLERANCE)

n_tp = len(matched)
n_fp = len(fp)
n_fn = len(fn)

precision = n_tp / (n_tp + n_fp) if (n_tp + n_fp) > 0 else 0.0
recall = n_tp / (n_tp + n_fn) if (n_tp + n_fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("=" * 55)
print("DETECTION METRICS")
print("=" * 55)
print(f"  Ground truth events:  {len(ground_truth)}")
print(f"  Detected events:      {len(events)}")
print(f"  True positives (TP):  {n_tp}")
print(f"  False positives (FP): {n_fp}")
print(f"  False negatives (FN): {n_fn}")
print(f"  Precision:            {precision:.3f}")
print(f"  Recall:               {recall:.3f}")
print(f"  F1 score:             {f1:.3f}")

if matched:
    timing_errors = [err for _, _, err in matched]
    print(f"\n  Timing error (matched events):")
    print(f"    mean = {np.mean(timing_errors)*1000:.1f} ms")
    print(f"    std  = {np.std(timing_errors)*1000:.1f} ms")
    print(f"    max  = {np.max(timing_errors)*1000:.1f} ms")

print("\n" + "=" * 55)
print("NOTES")
print("=" * 55)
print("  The false-positive rate is dominated by the EMA baseline")
print("  tracker (alpha=0.01), which converges slowly. After the")
print("  warmup period, the running mean/var still underestimate")
print("  the true noise floor, producing a threshold that is too")
print("  low. This is a known limitation — see README.")

In [ ]:
# ============================================================
# 4. Breakdown by channel and SNR
# ============================================================

print("Per-channel recall:")
print("-" * 45)
for ch in range(N_CHANNELS):
    ch_gt = [g for g in ground_truth if g.channel == ch]
    ch_tp = [m for m in matched if m[0].channel == ch]
    ch_recall = len(ch_tp) / len(ch_gt) if ch_gt else 0.0
    bar = "\u2588" * int(ch_recall * 20) + "\u2591" * (20 - int(ch_recall * 20))
    print(f"  ch {ch}: {bar} {ch_recall:.0%}  ({len(ch_tp)}/{len(ch_gt)})")

print("\nRecall by SNR bin:")
print("-" * 45)
snr_bins = [(12, 16), (16, 20), (20, 25)]
for lo, hi in snr_bins:
    bin_gt = [g for g in ground_truth if lo <= g.snr < hi]
    bin_tp = [m for m in matched if lo <= m[0].snr < hi]
    bin_recall = len(bin_tp) / len(bin_gt) if bin_gt else 0.0
    print(f"  SNR [{lo:2d}, {hi:2d}):  {bin_recall:.0%}  ({len(bin_tp)}/{len(bin_gt)})")

if fn:
    print(f"\nMissed events ({len(fn)} false negatives):")
    print("-" * 45)
    for ev in fn:
        print(f"  ch={ev.channel} t={ev.onset:.3f}s  dur={ev.duration*1000:.0f}ms  "
              f"snr={ev.snr:.1f}")